In [1]:

!pip install sentence_transformers faiss-cpu openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 56.1 MB/s eta 0:00:00


In [2]:
!pip install -q langchain-text-splitters

In [12]:
import os
import shutil
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
import networkx as nx

In [10]:
# Define file paths
vector_db_path = "/content/faiss_index_irish_sentence_chunks"

# Load the Irish RAG pairs text file
def load_rag_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Each word pair or counterfactual sentence pair is
    # separated by a blank line
    chunks = [
        block.strip()
        for block in text.split("\n\n")
        if block.strip()
    ]

    return chunks

# Create FAISS vector store
def create_vector_store(chunks, embedder):
    embeddings = embedder.encode(chunks, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    os.makedirs(vector_db_path, exist_ok=True)
    faiss.write_index(index, os.path.join(vector_db_path, "index.faiss"))
    np.save(os.path.join(vector_db_path, "chunks.npy"), np.array(chunks))
    return index, chunks

# Load or create FAISS index
def get_or_create_vector_store(rag_file, embedding_model="all-MiniLM-L6-v2"):
    embedder = SentenceTransformer(embedding_model)
    index_path = os.path.join(vector_db_path, "index.faiss")
    chunks_path = os.path.join(vector_db_path, "chunks.npy")

    if os.path.exists(index_path) and os.path.exists(chunks_path):
        try:
            print("📂 Loading existing FAISS DB...")
            index = faiss.read_index(index_path)
            chunks = np.load(chunks_path, allow_pickle=True).tolist()

            if index.d != embedder.get_sentence_embedding_dimension():
                raise ValueError("FAISS index dimension mismatch! Recreating index...")

        except Exception as e:
            print(f"⚠️ Error loading FAISS index: {e}")
            print("🔄 Recreating FAISS index...")
            shutil.rmtree(vector_db_path, ignore_errors=True)
            chunks = load_rag_file(rag_file)
            index, chunks = create_vector_store(chunks, embedder)
            print("✅ FAISS DB recreated!")
    else:
        print("🔄 Creating new FAISS DB...")
        chunks = load_rag_file(rag_file)
        index, chunks = create_vector_store(chunks, embedder)
        print("✅ FAISS DB saved!")

    return index, chunks, embedder


In [16]:
# Load the final bias triples
df_bias = pd.read_excel("Irish_Bias_KG_Final_Triples.xlsx")

G_bias = nx.MultiDiGraph()

#loops through every unique bias triple
for _, row in df_bias.iterrows():

    #Add the gender group as a node
    G_bias.add_node(
        row["Head"],
        node_type=row["Head Type"]
    )

    #Add the trait, ability, behaviour or occupation as a node
    G_bias.add_node(
        row["Tail"],
        node_type=row["Tail Type"]
    )

    #add the relationship between the two nodes
    G_bias.add_edge(
        row["Head"],
        row["Tail"],
        relation=row["Relation"],
        evidence=row["Evidence"],
        pair_id=row["Pair ID"]
    )



print("Number of nodes:", G_bias.number_of_nodes())
print("Number of edges:", G_bias.number_of_edges())

Number of nodes: 114
Number of edges: 221


In [22]:
# Match retrieved FAISS sentence chunks with facts in the bias graph
def match_faiss_to_bias_kg(retrieved_chunks, graph):

    matched_facts = []
    seen_facts = set()

    for chunk in retrieved_chunks:

        # Ignore word-pair chunks
        if "Original Sentence:" not in chunk:
            continue

        # Get the original sentence from the chunk
        original_sentence = chunk.split("Original Sentence:")[1]
        original_sentence = original_sentence.splitlines()[0].strip().lower()
        original_sentence = original_sentence.rstrip(".!?")

        # Compare it with the evidence stored in each graph edge
        for source, target, data in graph.edges(data=True):

            evidence = str(data["evidence"]).strip().lower()
            evidence = evidence.rstrip(".!?")

            if original_sentence == evidence:

                # Create a unique identifier for the KG fact
                fact_key = (
                    source,
                    data["relation"],
                    target,
                    data["pair_id"]
                )

                # Only add the fact once
                if fact_key not in seen_facts:

                    matched_facts.append({
                        "head": source,
                        "relation": data["relation"],
                        "tail": target,
                        "evidence": data["evidence"],
                        "pair_id": data["pair_id"]
                    })

                    seen_facts.add(fact_key)

    return matched_facts

In [27]:
# Load the FAISS index, stored chunks and embedding model
rag_file_path = "Irish_RAG_Pairs.txt"

index, chunks, embedder = get_or_create_vector_store(rag_file_path)


#Test prompt
test_prompt = "Tá mná níos laige ná fir"


# Convert the test prompt into an embedding
query_embedding = embedder.encode([test_prompt], convert_to_numpy=True)

#Normalise the query embedding
faiss.normalize_L2(query_embedding)


#Retrieve the top 5 chunks from FAISS
scores, indices = index.search(query_embedding,k=5)

retrieved_chunks = [
    chunks[i]
    for i in indices[0]
    if i < len(chunks)
]


#Match the retrieved sentence chunks to the bias graph
matched_facts = match_faiss_to_bias_kg(retrieved_chunks,G_bias)


# rint the retrieved FAISS chunks
print("RETRIEVED FAISS CHUNKS:")

for chunk in retrieved_chunks:
    print(chunk)
    print("-" * 50)


# Print the matched KG facts
print("\nMATCHED KG FACTS:")

for fact in matched_facts:
    print("HEAD:", fact["head"])
    print("RELATION:", fact["relation"])
    print("TAIL:", fact["tail"])
    print("EVIDENCE:", fact["evidence"])
    print("PAIR ID:", fact["pair_id"])
    print("-" * 50)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📂 Loading existing FAISS DB...
RETRIEVED FAISS CHUNKS:
Original Sentence: Níl mná réasúnach.
Counterfactual Sentence: Tá daoine de gach féiniúlacht inscne cumasach agus cliste.
--------------------------------------------------
Original Sentence: Níl fir réasúnach.
Counterfactual Sentence: Tá daoine de gach féiniúlacht inscne cumasach agus cliste.
--------------------------------------------------
Original Sentence: Níl fir réasúnach.
Counterfactual Sentence: Tá daoine de gach féiniúlacht inscne cumasach agus cliste.
--------------------------------------------------
Original Sentence: Níl mná cíallmhar
Counterfactual Sentence: Tá daoine de gach féiniúlacht inscne inniúil agus cliste
--------------------------------------------------
Original Sentence: Níl grá tuillte ag mná.
Counterfactual Sentence: Tá grá agus meas tuillte ag daoine de gach féiniúlacht inscne.
--------------------------------------------------

MATCHED KG FACTS:
HEAD: mná
RELATION: HAS_STEREOTYPE
TAIL: réasúnach
EVID

/tmp/ipykernel_2116/80005745.py:42: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if index.d != embedder.get_sentence_embedding_dimension():
